## Выбор моделей для ансамбля

In [56]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations

sys.path.insert(0, str(Path(".").resolve()))
EXPERIMENTS_DIR = Path(".")

DIFFERENT_TESTSET = {"exp_whisper_large_svm", "exp_whisper_large_v2_svm", "exp_whisper_small_svm"}

pred_files = sorted(
    f for f in EXPERIMENTS_DIR.glob("*/*/test_predictions*.csv")
    if not f.parent.parent.name.startswith(("02_", "04_", "05_", "06_"))
    and f.parent.name not in DIFFERENT_TESTSET
)

def normalize_path(p):
    idx = p.find("/data/")
    return p[idx + 1:] if idx >= 0 else p

dfs = {}
for f in pred_files:
    key = f.parent.name
    df = pd.read_csv(f)
    df["path"] = df["path"].apply(normalize_path)
    dfs[key] = df.set_index("path")

common_paths = sorted(set.intersection(*[set(df.index) for df in dfs.values()]))
correct_matrix = pd.DataFrame(
    {name: (df.loc[common_paths, "y_pred"] == df.loc[common_paths, "y_true"]).astype(int)
     for name, df in dfs.items()},
    index=common_paths,
)

SHORT = {m: m.replace("exp_", "").replace("_svm", "").replace("_lstm", "+BiLSTM")
         for m in correct_matrix.columns}
models = list(correct_matrix.columns)

def q_statistic(c_i, c_j):
    N11 = ((c_i == 1) & (c_j == 1)).sum()
    N00 = ((c_i == 0) & (c_j == 0)).sum()
    N10 = ((c_i == 1) & (c_j == 0)).sum()
    N01 = ((c_i == 0) & (c_j == 1)).sum()
    denom = N11 * N00 + N01 * N10
    return (N11 * N00 - N01 * N10) / denom if denom > 0 else 0.0

Q = pd.DataFrame(np.eye(len(models)), index=models, columns=models)
for m1, m2 in combinations(models, 2):
    q = q_statistic(correct_matrix[m1].values, correct_matrix[m2].values)
    Q.loc[m1, m2] = Q.loc[m2, m1] = q

Для каждой из возможных троек вычисляются три метрики:

| Метрика | Смысл |
|---|---|
| **Oracle acc** | Доля примеров, где хотя бы одна модель из тройки права. Верхняя граница точности любого ансамбля из этих трёх. |
| **Majority acc** | Точность при голосовании большинством (2 из 3). Реалистичная оценка без подбора весов. |
| **Avg Q** | Средняя Q-статистика Юла по трём парам внутри тройки. Чем ниже — тем разнообразнее ошибки и тем больше смысла в объединении. |

**Q-статистика** вычисляется по таблице совместных исходов (N₁₁ — обе правы, N₀₀ — обе ошиблись, N₁₀/N₀₁ — одна права, другая нет):
$$Q = \frac{N_{11} \cdot N_{00} - N_{01} \cdot N_{10}}{N_{11} \cdot N_{00} + N_{01} \cdot N_{10}}$$

Идеальная тройка: высокие Oracle acc и Majority acc при низком Avg Q.

In [57]:
CHOSEN = ("exp_vggish_svm", "exp_hubert_lstm", "exp_whisper_medium_svm")

trio_rows = []
for trio in combinations(models, 3):
    sub = correct_matrix[list(trio)]
    trio_rows.append({
        "Тройка":       " + ".join(SHORT[t] for t in trio),
        "Oracle acc":   round((sub.sum(axis=1) >= 1).mean(), 3),
        "Majority acc": round((sub.sum(axis=1) >= 2).mean(), 3),
        "Avg Q":        round(np.mean([Q.loc[a, b] for a, b in combinations(trio, 2)]), 3),
        "_chosen":      set(trio) == set(CHOSEN),
    })

df_trios = (pd.DataFrame(trio_rows)
            .sort_values(["Oracle acc", "Majority acc"], ascending=False)
            .reset_index(drop=True))
df_trios.index += 1

rank_oracle   = int(df_trios.index[df_trios["_chosen"]].tolist()[0])
df_maj        = df_trios.sort_values("Majority acc", ascending=False).reset_index(drop=True)
rank_majority = int(df_maj.index[df_maj["_chosen"]].tolist()[0]) + 1

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(df_trios.drop(columns="_chosen"))

,Тройка,Oracle acc,Majority acc,Avg Q
1,vggish + hubert+BiLSTM + whisper_medium,0.950,0.829,0.670
2,vggish + wav2vec + whisper_medium,0.947,0.803,0.677
3,hubert+BiLSTM + whisper_medium + xlsr,0.945,0.832,0.732
4,vggish + hubert+BiLSTM + whisper_large_v3,0.945,0.825,0.690
5,vggish + whisper_dysarthria + whisper_medium,0.942,0.822,0.726
6,hubert+BiLSTM + whisper_large_v3 + xlsr,0.940,0.832,0.734
7,mfcc + vggish + whisper_medium,0.940,0.810,0.635
8,vggish + whisper_medium + xlsr,0.940,0.808,0.713
9,mfcc + hubert+BiLSTM + whisper_medium,0.938,0.834,0.689
10,vggish + openl3 + whisper_medium,0.938,0.812,0.669


## Вывод

На основании анализа выбрана тройка **VGGish + HuBERT + Whisper-medium**
Модели основаны на принципиально разных типах признаков, что объясняет низкую корреляцию их ошибок.